# LangChain Document Loaders

Document Loaders form the foundational **Extract** phase of Retrieval-Augmented Generation (RAG) and LLM ETL pipelines.
They normalize incoming data from various sources (PDF, TXT, DOCX, etc.) into the standard LangChain `Document` schema:
- `page_content`: The extracted text.
- `metadata`: Dictionary containing source URI, page number, title, etc.


In [5]:
import os
import tempfile
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    DirectoryLoader,
)

load_dotenv()
print("Environment and loaders imported successfully!")


Environment and loaders imported successfully!


## 1. TextLoader
Loads plain text files into a single `Document` object.

In [6]:
def load_text_file():
    with tempfile.NamedTemporaryFile(delete=False, suffix=".txt", mode="w", encoding="utf-8") as temp_file:
        temp_file.write("Hello, this is a sample text file.\nThis file is used to demonstrate LangChain TextLoader.")
        temp_file_path = temp_file.name

    try:
        loader = TextLoader(temp_file_path, encoding="utf-8")
        documents = loader.load()

        print(f"Loaded {len(documents)} document(s)")
        print(f"Content preview: {documents[0].page_content[:100]}...")
        print(f"Metadata: {documents[0].metadata}")
        return documents
    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

text_docs = load_text_file()


Loaded 1 document(s)
Content preview: Hello, this is a sample text file.
This file is used to demonstrate LangChain TextLoader....
Metadata: {'source': 'C:\\Users\\AMANSH~1\\AppData\\Local\\Temp\\tmpd3xuhrz7.txt'}


## 2. PyPDFLoader
Parses PDF documents page-by-page. Emits one `Document` instance per physical page, preserving page metadata for citations.

In [7]:
def get_docs_path() -> Path:
    """Resolves docs directory whether running from workspace root or subfolder."""
    candidates = [
        Path("docs"),
        Path("LLM (for Chart Bot )/docs"),
        Path.cwd() / "docs",
        Path.cwd().parent / "LLM (for Chart Bot )/docs",
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    return Path("docs").resolve()

def load_pdf(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    print(f"Loaded {len(documents)} page(s) from PDF: {Path(pdf_path).name}\n" + "=" * 50)
    for i, doc in enumerate(documents):
        preview = doc.page_content.strip()[:150].replace('\n', ' ')
        print(f"\n--- Page {i + 1} ---")
        print(f"Content Preview: {preview}...")
        print(f"Metadata: {doc.metadata}")
    return documents

docs_folder = get_docs_path()
target_pdf = docs_folder / "Understanding_LangChain_Document_Loaders.pdf"

if target_pdf.exists():
    pdf_docs = load_pdf(str(target_pdf))
else:
    print(f"File not found at: {target_pdf}")


Loaded 3 page(s) from PDF: Understanding_LangChain_Document_Loaders.pdf

--- Page 1 ---
Content Preview: LANGCHAIN CORE ARCHITECTURE SERIES Understanding LangChain Document Loaders A comprehensive technical guide to unstructured data ingestion, interface ...
Metadata: {'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-20T03:33:31+00:00', 'title': 'Understanding LangChain Document Loaders', 'moddate': '2026-10-20T03:33:31+00:00', 'source': 'E:\\LANGCHAIN\\LLM (for Chart Bot )\\docs\\Understanding_LangChain_Document_Loaders.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}

--- Page 2 ---
Content Preview: 4 Taxonomy of Document Loaders LangChain organizes its extensive library of over 160 native loaders into distinct categories based on data provenance,...
Metadata: {'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-20T03:33:31+00:00', 'title': 'Understanding LangChain Document Loaders', 'moddate': '2026-10-20T03:33:31+00:00', 'so

## 3. DirectoryLoader (Batch Ingestion)
Crawls an entire filesystem tree matching a glob pattern (e.g. `*.pdf`) and delegates file decoding to `PyPDFLoader`.

In [9]:
def load_directory(dir_path: Path, glob_pattern: str = "*.pdf"):
    loader = DirectoryLoader(
        path=str(dir_path),
        glob=glob_pattern,
        loader_cls=PyPDFLoader,
        use_multithreading=True,
        max_concurrency=4
    )
    documents = loader.load()
    print(f"DirectoryLoader loaded {len(documents)} document(s) matching '{glob_pattern}' from '{dir_path.name}'")
    return documents

if docs_folder.exists():
    dir_docs = load_directory(docs_folder, "*.pdf")


DirectoryLoader loaded 3 document(s) matching '*.pdf' from 'docs'
